# Phase 1 — Prefix-LM run (10-hour Kaggle budget, 2× T4)



### Speed engineering in this notebook
| Change vs. previous notebook | Why it is faster |
|---|---|
| Corpus tokenized once into flat `uint16` files, read with `np.memmap` | no tokenization inside the training loop (was single-threaded, on the critical path) |
| `torchrun` + `DistributedDataParallel` instead of `nn.DataParallel` | one process per GPU, no model replication or output gathering on GPU 0 every step |
| `no_sync()` during gradient accumulation | gradients are all-reduced once per optimizer step, not once per micro-batch |
| fp16 autocast + fused AdamW, TF32/cudnn flags | fewer kernels, tensor-core matmuls |
| Schedule auto-sized from measured step time | LR decay finishes exactly when the time budget ends (no half-finished cosine) |
| Length-sorted, padding-free generation batches on both GPUs (+ KV cache if `model.py` provides it) | generation in minutes instead of ~3 h |
| Checkpoint every 20 min + automatic resume | a crash or timeout costs ≤ 20 min |

### Running it (one *Save & Run All* = train + `submission.zip`)
1. Attach: your tokenizer (vocab.json + merges.txt) and, ideally, the pre-tokenized `tokens/` dataset.
2. Run cells top to bottom once interactively up to the model check + smoke test to catch errors early.
3. *Save Version → Save & Run All (Commit)*. Order: model check → smoke test (≈5 min) → training →
   local eval + decoding sweep → test generation → **`/kaggle/working/submission.zip`** (validated).
   Any failure (model check, smoke test, training) stops the run instead of silently continuing.
4. Tip: run the pre-tokenization (cells 1–7) in a **CPU-only** notebook first and attach its output — no GPU quota used.

Cross-architecture validation comparison is deferred to Phase 2; every checkpoint needed for it is kept in `/kaggle/working/run/`.

In [ ]:
# 1 — session clock, deps, secrets
import time, os, json, glob, shutil
SESSION_START = time.time()
!pip install -q ijson
import torch
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])


try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)

## 2 — Configuration
Every `TODO` is your decision. The notebook refuses to start until none are left.

## Prefix-LM run: what was filled in, and what is held equal to arch2

| Setting | arch2 (earlier notebook) | This run | Why |
|---|---|---|---|
| Attention | causal | **prefix-LM** (bidirectional prompt, causal continuation) | the experiment |
| d_model / layers / heads / d_ff | 768 / 12 / 12 / 3072 | same | comparable capacity (~110M params) |
| MLP, norm, biases, tied embeddings | GELU, pre-LN, yes, yes | same | |
| Positions | learned absolute | RoPE (`MODEL["pos"]="learned"` reverts) | relative positions, no extra params |
| Dropout | 0.1 | 0.0 | one pass over the data, model underfits |
| Tokenizer, ctx_len | own 32k BPE, 1024 | same | |
| Tokens / optimizer step | 8 × 1024 × 16 = 131,072 | 8 × 1024 × 8 × 2 GPUs = 131,072 | |
| LR, schedule, warmup | 3e-4 → 3e-5 cosine, 1000 | same (length auto-sized to the time budget) | |
| AdamW β, weight decay, clip | (0.9, 0.95), 0.1, 1.0 | same; norms/biases excluded from decay | |
| Data mix SC / HC / Acts / Trib | 0.4 / 0.3 / 0.2 / 0.1 | same | |
| Validation | 300 SC windows, all tokens | 48 windows × 4 sources, prefix 0.75, last 256 tokens scored | matches the task: predict the continuation given ~768 tokens |

**Validation comparison with other architectures** is done later (Phase 2) by re-scoring saved checkpoints on the same windows; the val loss printed by the old notebooks is not comparable to this one.

In [ ]:
# 3 — CONFIG  (Prefix-LM run; everything that decides comparability with arch2 is marked [=arch2])
TODO = "TODO"

def _find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return hits[0] if hits else None


_meta = _find_input("tokens/meta.json")
_prev_ckpt = _find_input("run/prefixlm_step*.pt")

RUN = dict(
    run_name      = "prefixlm",
    session_start = SESSION_START,
    session_hours = 9.5,
    reserve_minutes_after_train = 75,
    token_dir     = os.path.dirname(_meta) if _meta else "/kaggle/working/tokens",
    tokenizer_dir = "/kaggle/working/tokenizer",
    out_dir       = "/kaggle/working/run",
    resume_dirs   = [os.path.dirname(_prev_ckpt)] if _prev_ckpt else [],
    val_windows_per_source = 48,
    eval_ckpt_overhead_frac = 0.04,
    smoke_test    = False,
)
print("token_dir:", RUN["token_dir"], "| resume_dirs:", RUN["resume_dirs"])

MODEL = dict(
    d_model=768, n_layers=12, n_heads=12, d_ff=3072,
    tie_embeddings=True, bias=True,
    pos="rope",
    rope_base=10000.0,
    dropout=0.0,
)

SOURCES = ["supreme_court", "high_courts", "acts", "tribunals"]
MIX = dict(supreme_court=0.40, high_courts=0.30, acts=0.20, tribunals=0.10)

TRAIN = dict(
    ctx_len=1024,
    micro_batch=8, grad_accum=8,

    lr_max=3e-4, lr_min=3e-5, warmup_steps=1000,
    schedule="cosine",
    decay_frac=None,
    weight_decay=0.1, wd_exclude_1d=True,
    beta1=0.9, beta2=0.95, grad_clip=1.0,
    prefix_mode="random",
    prefix_frac_min=0.30, prefix_frac_max=0.80,
    val_prefix_mode="fixed", val_prefix_frac=0.75,
    mix=MIX,
    max_tokens=None,
    amp_dtype="fp16",
    compile=False, grad_ckpt=False, find_unused=False,
    log_every_steps=20, val_every_min=30, ckpt_every_min=20,
)

GEN = dict(
    checkpoint="best",
    max_new_tokens=256,
    batch_size=32,
    temperature=0,
    top_k=0, top_p=1.0, rep_penalty=1.0,
    no_repeat_ngram=4, ngram_scope="generated",
    seed=0,
)


TOTAL_BUDGET = 1_500_000_000
PRETOK = dict(
    budgets={s: int(TOTAL_BUDGET * MIX[s]) for s in SOURCES},
    keep_every=1, val_permille=5, local_pairs_per_source=100,
)

CFG = dict(run=RUN, model=MODEL, train=TRAIN, gen=GEN, pretok=PRETOK)

def find_todos(obj, path=""):
    if obj == TODO: return [path]
    if isinstance(obj, dict): return sum((find_todos(v, f"{path}.{k}") for k, v in obj.items()), [])
    return []

missing = find_todos(CFG)
if missing:
    print("Still TODO:", *missing, sep="\n  ")
else:
    json.dump(CFG, open("run_config.json", "w"), indent=1)
    print("run_config.json written")

In [ ]:
# 4 — tokenizer (your own BPE from the earlier run; attach it as an input dataset)
os.makedirs(RUN["tokenizer_dir"], exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    dst = os.path.join(RUN["tokenizer_dir"], fn)
    if not os.path.exists(dst):
        hits = glob.glob(f"/kaggle/input/**/{fn}", recursive=True)
        assert hits, f"{fn} not found under /kaggle/input — attach the dataset/output holding your tokenizer"
        shutil.copy(hits[0], dst); print("tokenizer file:", hits[0])
from tokenizers import ByteLevelBPETokenizer
_tok = ByteLevelBPETokenizer(vocab=f"{RUN['tokenizer_dir']}/vocab.json", merges=f"{RUN['tokenizer_dir']}/merges.txt")
print("vocab:", _tok.get_vocab_size(), "| </s> id:", _tok.token_to_id("</s>"))

In [ ]:
%%writefile pretokenize.py
"""pretokenize.py — one-time data prep for all Phase-1 runs (ancillary code, not model code)."""
import argparse, json, os, random, time, zlib

import numpy as np


def parse_kv(items, cast=str):
    out = {}
    for it in items or []:
        k, v = it.split("=", 1)
        out[k.strip()] = cast(v.replace("_", "")) if cast is int else cast(v)
    return out


def is_heldout(text, permille):

    return zlib.crc32(text[:2000].encode("utf-8", "ignore")) % 1000 < permille


def load_tokenizer(tok_dir):
    from tokenizers import ByteLevelBPETokenizer
    tok = ByteLevelBPETokenizer(
        vocab=os.path.join(tok_dir, "vocab.json"),
        merges=os.path.join(tok_dir, "merges.txt"),
    )
    return tok


class BinWriter:

    def __init__(self, path):
        self.f = open(path, "wb")
        self.n = 0

    def write(self, ids_list):
        if not ids_list:
            return
        arr = np.fromiter((t for ids in ids_list for t in ids), dtype=np.uint16)
        arr.tofile(self.f)
        self.n += arr.size

    def close(self):
        self.f.close()


def process_source(name, path, budget, tok, eos_id, args, local_pairs, rng):
    import ijson

    tr = BinWriter(os.path.join(args.out, f"train_{name}.bin"))
    va = BinWriter(os.path.join(args.out, f"val_{name}.bin"))
    stats = dict(docs_seen=0, docs_train=0, docs_val=0, docs_short=0, docs_skipped_stride=0)

    batch_tr, batch_va = [], []
    t0 = time.time()

    def flush():
        for batch, writer in ((batch_tr, tr), (batch_va, va)):
            if batch:
                enc = tok.encode_batch(batch)
                writer.write([e.ids + [eos_id] for e in enc])
                batch.clear()

    with open(path, "rb") as f:
        for i, rec in enumerate(ijson.items(f, "item")):
            text = rec.get(args.text_key, "") if isinstance(rec, dict) else ""
            if not isinstance(text, str) or len(text) < args.min_chars:
                stats["docs_short"] += 1
                continue
            if args.keep_every > 1 and i % args.keep_every != 0:
                stats["docs_skipped_stride"] += 1
                continue
            stats["docs_seen"] += 1

            if is_heldout(text, args.val_permille):
                stats["docs_val"] += 1
                if va.n < args.val_budget_per_source:
                    batch_va.append(text)
                words = text.split()
                need = args.prompt_words + args.cont_words
                if len(words) >= need and len(local_pairs[name]) < args.local_pairs_per_source:
                    s = rng.randint(0, len(words) - need)
                    local_pairs[name].append((
                        " ".join(words[s:s + args.prompt_words]),
                        " ".join(words[s + args.prompt_words:s + need]),
                    ))
            else:
                stats["docs_train"] += 1
                batch_tr.append(text)

            if len(batch_tr) >= args.batch_docs or len(batch_va) >= args.batch_docs:
                flush()
                if stats["docs_seen"] % (args.batch_docs * 20) == 0:
                    rate = tr.n / max(1e-9, time.time() - t0)
                    print(f"  [{name}] train={tr.n/1e6:8.1f}M tok  val={va.n/1e6:6.2f}M  "
                          f"docs={stats['docs_seen']:,}  {rate/1e3:.0f}K tok/s", flush=True)
            if tr.n >= budget:
                break
    flush()
    tr.close(); va.close()
    stats.update(train_tokens=tr.n, val_tokens=va.n, seconds=round(time.time() - t0, 1))
    print(f"  [{name}] DONE train={tr.n/1e6:.1f}M val={va.n/1e6:.2f}M "
          f"in {stats['seconds']/60:.1f} min", flush=True)
    return stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tok_dir", required=True, help="folder with vocab.json + merges.txt")
    ap.add_argument("--out", required=True)
    ap.add_argument("--file", action="append", required=True, help="name=/path/to/file.json")
    ap.add_argument("--budget", action="append", required=True,
                    help="name=max_train_tokens (your data-mix choice)")
    ap.add_argument("--text_key", default="text")
    ap.add_argument("--min_chars", type=int, default=100)
    ap.add_argument("--keep_every", type=int, default=1,
                    help="keep every k-th doc to spread sampling across a file (1 = all)")
    ap.add_argument("--val_permille", type=int, default=5, help="held-out docs per 1000")
    ap.add_argument("--val_budget_per_source", type=int, default=2_000_000)
    ap.add_argument("--local_pairs_per_source", type=int, default=100)
    ap.add_argument("--prompt_words", type=int, default=512)
    ap.add_argument("--cont_words", type=int, default=128)
    ap.add_argument("--batch_docs", type=int, default=256)
    ap.add_argument("--eos_token", default="</s>")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    files = parse_kv(args.file)
    budgets = parse_kv(args.budget, int)
    missing = set(files) - set(budgets)
    if missing:
        raise SystemExit(f"No --budget given for: {sorted(missing)}")
    os.makedirs(args.out, exist_ok=True)

    tok = load_tokenizer(args.tok_dir)
    vocab = tok.get_vocab_size()
    if vocab > 65535:
        raise SystemExit(f"vocab {vocab} does not fit uint16")
    eos_id = tok.token_to_id(args.eos_token)
    if eos_id is None:
        raise SystemExit(f"EOS token {args.eos_token!r} not in vocab")

    rng = random.Random(args.seed)
    local_pairs = {n: [] for n in files}
    meta = dict(settings=vars(args), vocab_size=vocab, eos_id=eos_id, dtype="uint16", sources={})

    for name, path in files.items():
        print(f"Processing {name}: {path} (budget {budgets[name]/1e6:.0f}M train tokens)")
        meta["sources"][name] = process_source(name, path, budgets[name], tok, eos_id,
                                               args, local_pairs, rng)

    test, ref, origin = {}, {}, {}
    k = 1
    for name, pairs in local_pairs.items():
        for p, r in pairs:
            test[str(k)] = {"text": p}
            ref[str(k)] = {"text": r}
            origin[str(k)] = name
            k += 1
    for fn, obj in (("local_test.json", test), ("local_ref.json", ref), ("local_origin.json", origin)):
        with open(os.path.join(args.out, fn), "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False)
    meta["local_pairs"] = {n: len(p) for n, p in local_pairs.items()}
    with open(os.path.join(args.out, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    tot = sum(s["train_tokens"] for s in meta["sources"].values())
    print(f"\nAll done. Train tokens: {tot/1e6:.1f}M | local test pairs: {len(test)}")
    print(f"Files in {args.out}: train_*.bin, val_*.bin, local_test.json, local_ref.json, meta.json")


if __name__ == "__main__":
    main()

In [ ]:
# 6 — pre-tokenize (skipped if the token files already exist, e.g. from an attached CPU-notebook dataset)
from huggingface_hub import hf_hub_download
REPO = "Exploration-Lab/CS-781-Capstone"
TEST_PATH = hf_hub_download(REPO, "Phase_1_Pretraining/Test/test.json", repo_type="dataset")

if os.path.exists(os.path.join(RUN["token_dir"], "meta.json")):
    print("Token files ready:", json.load(open(os.path.join(RUN["token_dir"], "meta.json")))["sources"].keys())
else:
    assert PRETOK["budgets"] != TODO, "set PRETOK['budgets'] in the config cell"
    args = f"--tok_dir {RUN['tokenizer_dir']} --out {RUN['token_dir']} "
    args += f"--keep_every {PRETOK['keep_every']} --val_permille {PRETOK['val_permille']} "
    args += f"--local_pairs_per_source {PRETOK['local_pairs_per_source']} "
    for name, budget in PRETOK["budgets"].items():
        path = hf_hub_download(REPO, f"Phase_1_Pretraining/Train/train_{name}.json", repo_type="dataset")
        args += f"--file {name}={path} --budget {name}={int(budget)} "
    t0 = time.time()
    print("running: python pretokenize.py", args)
    !python pretokenize.py {args} 2>&1
    print(f"pre-tokenization took {(time.time()-t0)/60:.1f} min")
assert os.path.exists(os.path.join(RUN["token_dir"], "meta.json")), (
    "PRE-TOKENIZATION FAILED: no meta.json in " + RUN["token_dir"] + " — the real error is printed above this line")
print("token files OK:", sorted(os.listdir(RUN["token_dir"])))

In [ ]:
# 6b — data + length checks (decides max_new_tokens and shows whether any source repeats)
import numpy as np
_meta = json.load(open(os.path.join(RUN["token_dir"], "meta.json")))
src_tok = {s: v["train_tokens"] for s, v in _meta["sources"].items()}
EST_TRAIN_TOKENS = 1.2e9
print(f"{'source':15s} {'train tok':>10s} {'mix':>5s} {'epochs @ est.':>14s}")
for s, n in src_tok.items():
    ep = MIX.get(s, 0) * EST_TRAIN_TOKENS / max(1, n)
    print(f"{s:15s} {n/1e6:9.0f}M {MIX.get(s,0):5.2f} {ep:13.2f}" + ("   <-- repeats; lower its mix weight" if ep > 2 else ""))

def pct(v): return " / ".join(f"{np.percentile(v, q):.0f}" for q in (50, 95, 99)) + f" / max {max(v)}"
loc_p = [len(_tok.encode(v["text"]).ids) for v in json.load(open(f"{RUN['token_dir']}/local_test.json")).values()]
loc_r = [len(_tok.encode(v["text"]).ids) for v in json.load(open(f"{RUN['token_dir']}/local_ref.json")).values()]
tst_p = [len(_tok.encode(v["text"]).ids) for v in json.load(open(TEST_PATH, encoding="utf-8")).values()]
print("\ntokens  p50 / p95 / p99 / max")
print("  local prompts (512 w)   :", pct(loc_p))
print("  local refs    (128 w)   :", pct(loc_r))
print("  official test prompts   :", pct(tst_p))
keep = TRAIN["ctx_len"] - GEN["max_new_tokens"]
print(f"\nmax_new_tokens={GEN['max_new_tokens']} covers {np.mean(np.array(loc_r) <= GEN['max_new_tokens'] - 8)*100:.1f}% of 128-word continuations")
print(f"prompt window {keep} tokens: {np.mean(np.array(tst_p) > keep)*100:.1f}% of test prompts are left-trimmed")

In [ ]:
%%writefile model.py
"""model.py — Prefix-LM transformer (arch: "prefixlm")."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

DEFAULTS = dict(
    d_model=768, n_layers=12, n_heads=12, d_ff=3072,
    pos="rope", rope_base=10000.0,
    dropout=0.0, bias=True, tie_embeddings=True,
)


# ----------------------------------------------------------------------------- mask
def prefix_lm_mask(T, prefix_lens, device, start=0):
    i = torch.arange(T, device=device)
    causal = i[None, :] <= i[:, None]
    in_prefix = (i + start)[None, None, :] < prefix_lens[:, None, None]
    return (causal[None] | in_prefix)[:, None]


# ----------------------------------------------------------------------------- rotary embedding
def rope_tables(head_dim, max_len, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
    f = torch.outer(torch.arange(max_len, dtype=torch.float32), inv)
    f = torch.cat([f, f], dim=-1)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    h = x.shape[-1] // 2
    rot = torch.cat([-x[..., h:], x[..., :h]], dim=-1)
    return x * cos + rot * sin


# ----------------------------------------------------------------------------- layers
class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.h, self.d = c["n_heads"], c["d_model"]
        assert self.d % self.h == 0
        self.hd = self.d // self.h
        self.qkv = nn.Linear(self.d, 3 * self.d, bias=c["bias"])
        self.proj = nn.Linear(self.d, self.d, bias=c["bias"])
        self.p_drop = c["dropout"]
        self.rope = c["pos"] == "rope"

    def forward(self, x, cos, sin, mask, kv=None, start=0):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.hd).permute(2, 0, 3, 1, 4)
        if self.rope:
            cos, sin = cos.to(q.dtype), sin.to(q.dtype)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        if kv is not None:
            cache, li = kv
            if cache.k[li] is None:
                shape = (B, self.h, cache.max_len, self.hd)
                cache.k[li] = torch.empty(shape, device=k.device, dtype=k.dtype)
                cache.v[li] = torch.empty(shape, device=v.device, dtype=v.dtype)
            kbuf, vbuf = cache.k[li], cache.v[li]
            kbuf[:, :, start:start + T] = k
            vbuf[:, :, start:start + T] = v
            k, v = kbuf[:, :, :start + T], vbuf[:, :, :start + T]
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=False,
            dropout_p=self.p_drop if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, self.d))


class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.fc1 = nn.Linear(c["d_model"], c["d_ff"], bias=c["bias"])
        self.fc2 = nn.Linear(c["d_ff"], c["d_model"], bias=c["bias"])
        self.act = nn.GELU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1 = nn.LayerNorm(c["d_model"])
        self.attn = Attention(c)
        self.ln2 = nn.LayerNorm(c["d_model"])
        self.mlp = MLP(c)
        self.drop = nn.Dropout(c["dropout"])

    def forward(self, x, cos, sin, mask, kv=None, start=0):
        x = x + self.drop(self.attn(self.ln1(x), cos, sin, mask, kv, start))
        x = x + self.drop(self.mlp(self.ln2(x)))
        return x


class KVCache:
    def __init__(self, n_layers, max_len):
        self.k, self.v = [None] * n_layers, [None] * n_layers
        self.max_len, self.pos = max_len, 0


# ----------------------------------------------------------------------------- model
class PrefixLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        V, D, L = c["vocab_size"], c["d_model"], c["n_layers"]
        self.max_len = c["max_seq_len"]
        self.tok_emb = nn.Embedding(V, D)
        self.pos_emb = nn.Embedding(self.max_len, D) if c["pos"] == "learned" else None
        self.drop = nn.Dropout(c["dropout"])
        self.blocks = nn.ModuleList([Block(c) for _ in range(L)])
        self.ln_f = nn.LayerNorm(D)
        self.head = nn.Linear(D, V, bias=False)
        if c["pos"] == "rope":
            cos, sin = rope_tables(D // c["n_heads"], self.max_len, c["rope_base"])
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.rope_cos = self.rope_sin = None
        self.grad_ckpt = False

        self.apply(self._init)
        for n, p in self.named_parameters():
            if n.endswith("attn.proj.weight") or n.endswith("mlp.fc2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * L))
        if c["tie_embeddings"]:
            self.head.weight = self.tok_emb.weight

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def set_gradient_checkpointing(self, flag: bool):
        self.grad_ckpt = bool(flag)


    def _hidden(self, input_ids, mask, cache=None, start=0):
        B, T = input_ids.shape
        assert start + T <= self.max_len, f"sequence {start + T} > max_seq_len {self.max_len}"
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            x = x + self.pos_emb(torch.arange(start, start + T, device=input_ids.device))
        x = self.drop(x)
        cos = self.rope_cos[start:start + T] if self.rope_cos is not None else None
        sin = self.rope_sin[start:start + T] if self.rope_sin is not None else None
        for li, blk in enumerate(self.blocks):
            kv = None if cache is None else (cache, li)
            if self.grad_ckpt and self.training and cache is None:
                x = checkpoint(blk, x, cos, sin, mask, use_reentrant=False)
            else:
                x = blk(x, cos, sin, mask, kv, start)
        return self.ln_f(x)

    def forward(self, input_ids, prefix_lens=None):
        B, T = input_ids.shape
        if prefix_lens is None:
            prefix_lens = torch.zeros(B, dtype=torch.long, device=input_ids.device)
        mask = prefix_lm_mask(T, prefix_lens.to(input_ids.device), input_ids.device)
        return self.head(self._hidden(input_ids, mask))

    # ------------------------------------------------------------------ KV-cached generation
    @torch.no_grad()
    def prefill(self, input_ids, prefix_lens):
        B, T = input_ids.shape
        dev = input_ids.device
        cache = KVCache(len(self.blocks), self.max_len)
        mask = prefix_lm_mask(T, prefix_lens.to(dev), dev)
        h = self._hidden(input_ids, mask, cache, 0)
        cache.pos = T
        return self.head(h[:, -1]), cache

    @torch.no_grad()
    def decode_step(self, next_ids, cache):


        h = self._hidden(next_ids, None, cache, cache.pos)
        cache.pos += next_ids.shape[1]
        return self.head(h[:, -1]), cache


def build_model(model_cfg: dict) -> nn.Module:
    c = {**DEFAULTS, **model_cfg}
    assert c["pos"] in ("rope", "learned"), c["pos"]
    return PrefixLM(c)

In [ ]:
# 7b — model.py self-check on GPU: mask picture, prefix isolation, bidirectionality, KV cache, memory/speed
import importlib, sys, torch
sys.path.insert(0, "."); import model as M; importlib.reload(M)
V = _tok.get_vocab_size()
m = M.build_model(dict(MODEL, vocab_size=V, max_seq_len=TRAIN["ctx_len"])).cuda()
n = sum(p.numel() for p in m.parameters())
print(f"Parameters: {n/1e6:.2f}M"); assert 50e6 <= n <= 300e6

print("Mask, T=6, prefix_len=3 (row = query i, column = key j):")
print(M.prefix_lm_mask(6, torch.tensor([3]), "cpu")[0, 0].int())

m.eval(); T, P = 96, 60
x = torch.randint(0, V, (2, T), device="cuda"); pl = torch.tensor([P, P], device="cuda")
with torch.no_grad():
    a = m(x, pl).float()
    x2 = x.clone(); x2[:, P + 5] = (x2[:, P + 5] + 1) % V; b = m(x2, pl).float()
    assert torch.allclose(a[:, :P + 5], b[:, :P + 5], atol=1e-4), "a suffix token leaked backwards"
    assert (a[:, P + 5:] - b[:, P + 5:]).abs().max() > 1e-4
    x3 = x.clone(); x3[:, P - 1] = (x3[:, P - 1] + 1) % V; c = m(x3, pl).float()
    assert (c[:, 0] - a[:, 0]).abs().max() > 1e-4, "prefix is not bidirectional"
    with torch.autocast("cuda", dtype=torch.float16):
        full = m(x, pl)[:, P - 1:T - 1].float()
        lg, cache = m.prefill(x[:, :P], pl); outs = [lg.float()]
        for t in range(P, T - 1):
            lg, cache = m.decode_step(x[:, t:t + 1], cache); outs.append(lg.float())
    err = (torch.stack(outs, 1) - full).abs().max().item()
    print(f"KV-cache vs full forward, max |diff| = {err:.4f}"); assert err < 0.05
print("mask / isolation / bidirectional / KV-cache checks passed")


m.train(); MB, CTX = TRAIN["micro_batch"], TRAIN["ctx_len"]
torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = None
for it in range(6):
    if it == 2: torch.cuda.synchronize(); t0 = time.time()
    xb = torch.randint(0, V, (MB, CTX), device="cuda")
    pb = torch.randint(int(.3 * CTX), int(.8 * CTX), (MB,), device="cuda")
    with torch.autocast("cuda", dtype=torch.float16):
        loss = torch.nn.functional.cross_entropy(m(xb, pb).float().view(-1, V), xb.view(-1))
    loss.backward(); m.zero_grad(set_to_none=True)
torch.cuda.synchronize()
tps = 4 * MB * CTX / (time.time() - t0)
peak = torch.cuda.max_memory_allocated() / 1e9
print(f"micro_batch {MB}: peak {peak:.1f} GB activations+weights (+~1.8 GB AdamW/DDP buckets in training) | "
      f"~{tps/1e3:.1f}K tok/s per GPU (no optimizer) -> ~{2*tps*0.85/1e3:.0f}K tok/s on 2 GPUs")
del m, loss, xb, pb, a, b, c, full, cache; torch.cuda.empty_cache()

In [ ]:
%%writefile dummy_model.py
"""dummy_model.py — PIPELINE TEST ONLY (used when RUN["smoke_test"] = True)."""
import torch.nn as nn


class _Dummy(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)
        self.out = nn.Linear(64, vocab_size)

    def forward(self, input_ids, prefix_lens):
        return self.out(self.emb(input_ids))


def build_model(model_cfg):
    return _Dummy(model_cfg["vocab_size"])

In [ ]:
%%writefile train.py
import contextlib, glob, json, math, os, sys, time

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP

CFG = json.load(open(sys.argv[1]))
RUN, MODEL_CFG, TR = CFG["run"], CFG["model"], CFG["train"]

if RUN.get("smoke_test"):
    import dummy_model as model_lib
else:
    import model as model_lib


# ----------------------------------------------------------------------------- setup
def setup_dist():
    if int(os.environ.get("WORLD_SIZE", "1")) > 1:
        dist.init_process_group("nccl")
        rank, world, local = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    torch.cuda.set_device(local)
    return rank, world, torch.device("cuda", local)


RANK, WORLD, DEV = setup_dist()
IS_MAIN = RANK == 0
DDP_ON = WORLD > 1


def log(*a):
    if IS_MAIN:
        print(*a, flush=True)


def bcast_float(x):
    t = torch.tensor([float(x)], device=DEV)
    if DDP_ON:
        dist.broadcast(t, 0)
    return t.item()


def allreduce_sum(*vals):
    t = torch.tensor([float(v) for v in vals], device=DEV, dtype=torch.float64)
    if DDP_ON:
        dist.all_reduce(t)
    return t.tolist()


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ----------------------------------------------------------------------------- time budget
SESSION_START = RUN["session_start"]
SMOKE = bool(RUN.get("smoke_test"))
if SMOKE:
    TRAIN_DEADLINE = time.time() + RUN.get("smoke_train_minutes", 3) * 60
else:
    TRAIN_DEADLINE = (SESSION_START + RUN["session_hours"] * 3600
                      - RUN["reserve_minutes_after_train"] * 60)
log(f"Train deadline in {(TRAIN_DEADLINE - time.time())/3600:.2f} h")

# ----------------------------------------------------------------------------- data
META = json.load(open(os.path.join(RUN["token_dir"], "meta.json")))
CTX = int(TR["ctx_len"])


class WindowSampler:

    def __init__(self, split, mix, seed):
        files = sorted(glob.glob(os.path.join(RUN["token_dir"], f"{split}_*.bin")))
        assert files, f"no {split}_*.bin files in {RUN['token_dir']}"
        self.names, self.arrs = [], []
        for f in files:
            a = np.memmap(f, dtype=np.uint16, mode="r")
            if a.size > CTX + 2:
                self.names.append(os.path.basename(f)[len(split) + 1:-4])
                self.arrs.append(a)
        if mix == "proportional":
            w = np.array([a.size for a in self.arrs], dtype=np.float64)
        else:
            w = np.array([float(mix.get(n, 0.0)) for n in self.names])
        assert w.sum() > 0, f"mix gives zero weight to all of {self.names}"
        self.w = w / w.sum()
        self.rng = np.random.default_rng(seed)

    def window(self, i):
        a = self.arrs[i]
        s = int(self.rng.integers(0, a.size - CTX - 1))
        return np.asarray(a[s:s + CTX + 1], dtype=np.int64)

    def batch(self, B):
        idx = self.rng.choice(len(self.arrs), size=B, p=self.w)
        return torch.from_numpy(np.stack([self.window(i) for i in idx]))


def prefix_lengths(B, mode, fmin, fmax, rng):
    if mode == "none":
        return np.zeros(B, dtype=np.int64)
    lo = int(round(fmin * CTX))
    if mode == "fixed":
        return np.full(B, lo, dtype=np.int64)
    if mode == "random":
        hi = int(round(fmax * CTX))
        return rng.integers(lo, hi + 1, size=B).astype(np.int64)
    raise ValueError(f"prefix_mode must be none|fixed|random, got {mode!r}")


def make_xy(chunk, plen):
    x, y = chunk[:, :-1].clone(), chunk[:, 1:].clone()
    pos = torch.arange(CTX)[None, :]
    y[pos < (torch.from_numpy(plen)[:, None] - 1)] = -100
    return x, y, torch.from_numpy(plen)


train_sampler = WindowSampler("train", TR["mix"], seed=1234 + RANK * 7919)
train_prefix_rng = np.random.default_rng(99 + RANK)
log(f"Sources: {dict(zip(train_sampler.names, np.round(train_sampler.w, 3)))}")


VAL_PER_SOURCE = 8 if SMOKE else int(RUN.get("val_windows_per_source", 48))
val_sampler = WindowSampler("val", "proportional", seed=4321)
VAL = []
vp_rng = np.random.default_rng(7)
for si, name in enumerate(val_sampler.names):
    for _ in range(VAL_PER_SOURCE):
        c = torch.from_numpy(val_sampler.window(si))[None]
        pl = prefix_lengths(1, TR["val_prefix_mode"], TR["val_prefix_frac"], TR["val_prefix_frac"], vp_rng)
        x, y, p = make_xy(c, pl)
        VAL.append((name, x, y, p))
VAL_SHARD = VAL[RANK::WORLD]

# ----------------------------------------------------------------------------- model
MODEL_CFG.setdefault("vocab_size", META["vocab_size"])
MODEL_CFG.setdefault("max_seq_len", CTX)
raw_model = model_lib.build_model(MODEL_CFG).to(DEV)
n_params = sum(p.numel() for p in raw_model.parameters())
log(f"Parameters: {n_params/1e6:.2f}M")
if not SMOKE:
    assert 50e6 <= n_params <= 300e6, "Phase-1 rule: model must be 50M-300M parameters"
if TR.get("grad_ckpt") and hasattr(raw_model, "set_gradient_checkpointing"):
    raw_model.set_gradient_checkpointing(True)

decay, no_decay = [], []
for n, p in raw_model.named_parameters():
    if p.requires_grad:
        (decay if (p.dim() >= 2 or not TR["wd_exclude_1d"]) else no_decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": TR["weight_decay"]},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=TR["lr_max"], betas=(TR["beta1"], TR["beta2"]), eps=1e-8, fused=True)
scaler = torch.amp.GradScaler("cuda", enabled=TR.get("amp_dtype", "fp16") == "fp16")
AMP_DTYPE = torch.float16 if TR.get("amp_dtype", "fp16") == "fp16" else torch.bfloat16

model = raw_model
if TR.get("compile"):
    model = torch.compile(model)
if DDP_ON:
    model = DDP(model, device_ids=[DEV.index], find_unused_parameters=bool(TR.get("find_unused", False)))


def unwrap():
    return raw_model


# ----------------------------------------------------------------------------- checkpoints
OUT = RUN["out_dir"]
os.makedirs(OUT, exist_ok=True)
NAME = RUN["run_name"]


def ckpt_files(d):
    return sorted(glob.glob(os.path.join(d, f"{NAME}_step*.pt")))


def save_ckpt(state, path):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)


def save_training_ckpt(step, tokens, total_steps, best_val):
    if not IS_MAIN:
        return
    path = os.path.join(OUT, f"{NAME}_step{step:07d}.pt")
    save_ckpt(dict(model=unwrap().state_dict(), optimizer=optimizer.state_dict(),
                   scaler=scaler.state_dict(), step=step, tokens=tokens,
                   total_steps=total_steps, best_val=best_val, cfg=CFG), path)
    for old in ckpt_files(OUT)[:-2]:
        os.remove(old)
    log(f"  saved {os.path.basename(path)}")


def save_model_only(tag, step, tokens, val):
    if IS_MAIN:
        save_ckpt(dict(model=unwrap().state_dict(), step=step, tokens=tokens, val_loss=val,
                       model_cfg=MODEL_CFG, n_params=n_params),
                  os.path.join(OUT, f"{NAME}_{tag}.pt"))


step, tokens, total_steps, best_val = 0, 0, None, float("inf")
RESUMED = False
cands = []
for d in [OUT] + list(RUN.get("resume_dirs", [])):
    cands += ckpt_files(d)
if cands:
    latest = max(cands, key=lambda p: int(p.split("_step")[-1][:-3]))
    ck = torch.load(latest, map_location=DEV, weights_only=False)
    unwrap().load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    step, tokens, total_steps, best_val = ck["step"], ck["tokens"], ck["total_steps"], ck["best_val"]
    log(f"Resumed from {latest} (step {step}, {tokens/1e6:.1f}M tokens)")
    RESUMED = True
    del ck

    train_sampler.rng = np.random.default_rng(1234 + RANK * 7919 + 1_000_003 * step)
    train_prefix_rng = np.random.default_rng(99 + RANK + 1_000_003 * step)
    log("Data sampler re-seeded for resume")

# ----------------------------------------------------------------------------- schedule
def lr_at(s, total):
    lr_max, lr_min, warm = TR["lr_max"], TR["lr_min"], TR["warmup_steps"]
    if s < warm:
        return lr_max * (s + 1) / warm
    if total is None:
        return lr_max
    t = min(1.0, (s - warm) / max(1, total - warm))
    kind = TR["schedule"]
    if kind == "cosine":
        return lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * t))
    if kind == "linear":
        return lr_max + (lr_min - lr_max) * t
    if kind == "constant":
        return lr_max
    if kind == "wsd":
        d = TR["decay_frac"]
        return lr_max if t < 1 - d else lr_max + (lr_min - lr_max) * (t - (1 - d)) / d
    raise ValueError(f"unknown schedule {kind!r}")


# ----------------------------------------------------------------------------- eval
@torch.no_grad()
def evaluate():
    model.eval()
    sums = {}
    for name, x, y, p in VAL_SHARD:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = unwrap()(x.to(DEV), p.to(DEV))
        l = F.cross_entropy(logits.float().view(-1, logits.size(-1)), y.to(DEV).view(-1),
                            ignore_index=-100, reduction="sum").item()
        n = (y != -100).sum().item()
        s = sums.setdefault(name, [0.0, 0])
        s[0] += l; s[1] += n
    names = val_sampler.names
    flat = []
    for n in names:
        flat += sums.get(n, [0.0, 0])
    flat = allreduce_sum(*flat)
    per = {n: flat[2 * i] / max(1, flat[2 * i + 1]) for i, n in enumerate(names)}
    tot = sum(flat[0::2]) / max(1, sum(flat[1::2]))
    model.train()
    return tot, per


# ----------------------------------------------------------------------------- train loop
MB, ACC = int(TR["micro_batch"]), int(TR["grad_accum"])
TOK_PER_STEP = MB * CTX * ACC * WORLD
log(f"Tokens/step: {TOK_PER_STEP:,}  (micro {MB} x ctx {CTX} x accum {ACC} x gpus {WORLD})")

csv_path = os.path.join(OUT, f"{NAME}_log.csv")
if IS_MAIN and not os.path.exists(csv_path):
    open(csv_path, "w").write("step,tokens,train_loss,lr,tok_per_s,elapsed_h,val_loss,val_per_source\n")


def csv(row):
    if IS_MAIN:
        open(csv_path, "a").write(row + "\n")


CALIB_AFTER = 10
CALIB_STEPS = 30
calib_t0 = None
run_t0 = time.time()
last_ckpt = time.time()
last_val = time.time()
win_t0, win_tok = time.time(), 0
model.train()
optimizer.zero_grad(set_to_none=True)
steps_this_run = 0

while True:

    loss_acc = torch.zeros((), device=DEV)
    for micro in range(ACC):
        chunk = train_sampler.batch(MB)
        pl = prefix_lengths(MB, TR["prefix_mode"], TR.get("prefix_frac_min", 0.0),
                            TR.get("prefix_frac_max", 0.0), train_prefix_rng)
        x, y, p = make_xy(chunk, pl)
        x, y, p = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True)
        sync_ctx = model.no_sync() if (DDP_ON and micro < ACC - 1) else contextlib.nullcontext()
        with sync_ctx:
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(x, p)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)).float(), y.view(-1),
                                       ignore_index=-100)
            scaler.scale(loss / ACC).backward()
        loss_acc += loss.detach() / ACC

    lr = lr_at(step, total_steps)
    for g in optimizer.param_groups:
        g["lr"] = lr
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), TR["grad_clip"])
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    step += 1
    steps_this_run += 1
    tokens += TOK_PER_STEP
    win_tok += TOK_PER_STEP


    if steps_this_run == CALIB_AFTER:
        torch.cuda.synchronize(); calib_t0 = time.time()
    if steps_this_run == CALIB_AFTER + CALIB_STEPS and (total_steps is None or RESUMED):
        torch.cuda.synchronize()
        sec_per_step = (time.time() - calib_t0) / CALIB_STEPS
        overhead = 1.0 - RUN.get("eval_ckpt_overhead_frac", 0.04)
        est = step + int((TRAIN_DEADLINE - time.time()) / sec_per_step * overhead)
        if TR.get("max_tokens"):
            est = min(est, int(TR["max_tokens"]) // TOK_PER_STEP)
        est = int(bcast_float(est))
        if RESUMED and total_steps is not None:
            if est < total_steps:
                log(f"Resume: only time for {est} of {total_steps} scheduled steps -> schedule shortened so LR still decays")
                total_steps = est
        else:
            total_steps = est
        log(f"Measured {sec_per_step:.2f}s/step ({TOK_PER_STEP/sec_per_step/1e3:.1f}K tok/s). "
            f"Schedule sized to {total_steps} steps = {total_steps*TOK_PER_STEP/1e6:.0f}M tokens.")

    now = time.time()
    if step % TR.get("log_every_steps", 20) == 0:
        l = allreduce_sum(loss_acc.item())[0] / WORLD
        tps = win_tok / max(1e-9, now - win_t0)
        log(f"step {step:6d} | loss {l:.4f} | lr {lr:.2e} | {tokens/1e6:8.1f}M tok | "
            f"{tps/1e3:5.1f}K tok/s | left {(TRAIN_DEADLINE-now)/3600:.2f}h")
        csv(f"{step},{tokens},{l:.5f},{lr:.3e},{tps:.0f},{(now-run_t0)/3600:.3f},,")
        win_t0, win_tok = now, 0


    flags = torch.tensor([
        float((now >= TRAIN_DEADLINE) or (total_steps is not None and step >= total_steps)),
        float(now - last_val >= TR.get("val_every_min", 30) * 60),
        float(now - last_ckpt >= TR.get("ckpt_every_min", 20) * 60)], device=DEV)
    if DDP_ON:
        dist.broadcast(flags, 0)
    stop, do_val, do_ckpt = (bool(f) for f in flags.tolist())

    if stop or do_val:
        v, per = evaluate()
        log(f"  [val] loss {v:.4f} ppl {math.exp(min(v, 20)):.2f} | " +
            " ".join(f"{k}={x:.3f}" for k, x in per.items()))
        csv(f"{step},{tokens},,{lr:.3e},,{(now-run_t0)/3600:.3f},{v:.5f},"
            + ";".join(f"{k}={x:.4f}" for k, x in per.items()))
        if v < best_val:
            best_val = v
            save_model_only("best", step, tokens, v)
            log(f"  new best -> {NAME}_best.pt")
        last_val = time.time()

    if stop or do_ckpt:
        save_training_ckpt(step, tokens, total_steps, best_val)
        last_ckpt = time.time()

    if stop:
        break

save_model_only("final", step, tokens, best_val)
log(f"Training finished: {step} steps, {tokens/1e6:.1f}M tokens, best val {best_val:.4f}")
if DDP_ON:
    dist.barrier()
    dist.destroy_process_group()


In [ ]:
%%writefile generate.py
import glob, json, os, re, sys, time, zipfile

import torch
import torch.distributed as dist
import torch.nn.functional as F

CFG = json.load(open(sys.argv[1]))
IN_PATH, OUT_PREFIX = sys.argv[2], sys.argv[3]
RUN, MODEL_CFG, GEN = CFG["run"], CFG["model"], CFG["gen"]
TARGET_WORDS = 128

if RUN.get("smoke_test"):
    import dummy_model as model_lib
else:
    import model as model_lib

if int(os.environ.get("WORLD_SIZE", "1")) > 1:
    dist.init_process_group("nccl")
    RANK, WORLD, LOCAL = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
else:
    RANK, WORLD, LOCAL = 0, 1, 0
torch.cuda.set_device(LOCAL)
DEV = torch.device("cuda", LOCAL)
torch.manual_seed(int(GEN.get("seed", 0)) + RANK)


def log(*a):
    if RANK == 0:
        print(*a, flush=True)


# ----------------------------------------------------------------------------- tokenizer + model
from tokenizers import ByteLevelBPETokenizer

tok = ByteLevelBPETokenizer(vocab=os.path.join(RUN["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(RUN["tokenizer_dir"], "merges.txt"))
CTX = int(CFG["train"]["ctx_len"])
MODEL_CFG.setdefault("vocab_size", tok.get_vocab_size())
MODEL_CFG.setdefault("max_seq_len", CTX)

ck_choice = GEN["checkpoint"]
ck_path = ck_choice if os.path.exists(ck_choice) else os.path.join(RUN["out_dir"], f"{RUN['run_name']}_{ck_choice}.pt")
model = model_lib.build_model(MODEL_CFG).to(DEV)
ck = torch.load(ck_path, map_location=DEV, weights_only=False)
model.load_state_dict(ck["model"])
model.eval()
log(f"Loaded {ck_path} (step {ck.get('step')}, val {ck.get('val_loss')})")
del ck
HAS_CACHE = hasattr(model, "prefill") and hasattr(model, "decode_step")
log("Decoding with KV cache (prefill/decode_step)" if HAS_CACHE else
    "No prefill/decode_step on the model: recomputing the full sequence each step (slower)")

# ----------------------------------------------------------------------------- sampling utils
MAX_NEW = int(GEN["max_new_tokens"])
PROMPT_MAX = CTX - MAX_NEW
assert PROMPT_MAX > 64, "ctx_len too small for max_new_tokens"
BAN = [tok.token_to_id(t) for t in ("<s>", "</s>", "<unk>", "<pad>", "<mask>") if tok.token_to_id(t) is not None]


class NgramBlocker:

    def __init__(self, n, prompt_ids, scope):
        self.n, self.seen = n, {}
        self.hist = list(prompt_ids[-(n - 1):]) if (n and scope != "all") else []
        if n and scope == "all":
            for i in range(len(prompt_ids) - n + 1):
                self.seen.setdefault(tuple(prompt_ids[i:i + n - 1]), set()).add(prompt_ids[i + n - 1])
            self.hist = list(prompt_ids[-(n - 1):])
        self.scope = scope
        self.gen_count = 0

    def banned(self):
        if not self.n or len(self.hist) < self.n - 1:
            return ()
        return self.seen.get(tuple(self.hist[-(self.n - 1):]), ())

    def push(self, t):
        if self.n:
            key = tuple(self.hist[-(self.n - 1):])

            if len(key) == self.n - 1 and (self.scope == "all" or self.gen_count >= self.n - 1):
                self.seen.setdefault(key, set()).add(t)
            self.hist.append(t)
            self.gen_count += 1


def pick_next(logits, gen_tokens, blockers):
    if BAN:
        logits[:, BAN] = -float("inf")
    rp = GEN["rep_penalty"]
    if rp and rp != 1.0 and gen_tokens.numel():
        sc = torch.gather(logits, 1, gen_tokens)
        logits.scatter_(1, gen_tokens, torch.where(sc < 0, sc * rp, sc / rp))
    for b, bl in enumerate(blockers):
        bad = bl.banned()
        if bad:
            logits[b, list(bad)] = -float("inf")
    T = GEN["temperature"]
    if not T:
        return logits.argmax(-1)
    logits = logits / T
    k = GEN.get("top_k") or 0
    if k:
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    p = GEN.get("top_p") or 1.0
    probs = F.softmax(logits, -1)
    if p < 1.0:
        sp, si = probs.sort(-1, descending=True)
        sp[(sp.cumsum(-1) - sp) > p] = 0.0
        probs = torch.zeros_like(probs).scatter_(1, si, sp)
    probs = probs / probs.sum(-1, keepdim=True)
    return torch.multinomial(probs, 1).squeeze(1)


def clean(text):
    return re.sub(r"\s+", " ", text).strip()


@torch.inference_mode()
def generate_batch(prompts_ids):
    B, L = len(prompts_ids), len(prompts_ids[0])
    ids = torch.tensor(prompts_ids, dtype=torch.long, device=DEV)
    plen = torch.full((B,), L, dtype=torch.long, device=DEV)
    blockers = [NgramBlocker(GEN.get("no_repeat_ngram") or 0, p, GEN.get("ngram_scope", "generated"))
                for p in prompts_ids]
    gen = torch.empty((B, 0), dtype=torch.long, device=DEV)
    done = torch.zeros(B, dtype=torch.bool, device=DEV)
    with torch.autocast("cuda", dtype=torch.float16):
        if HAS_CACHE:
            logits, cache = model.prefill(ids, plen)
        else:
            logits = model(ids, plen)[:, -1]
    for step in range(MAX_NEW):
        nxt = pick_next(logits.float(), gen, blockers)
        gen = torch.cat([gen, nxt[:, None]], 1)
        nl = nxt.tolist()
        for b, bl in enumerate(blockers):
            bl.push(nl[b])
        if (step + 1) % 16 == 0:
            for b in range(B):
                if not done[b] and len(tok.decode(gen[b].tolist()).split()) > TARGET_WORDS:
                    done[b] = True
            if bool(done.all()):
                break
        if step == MAX_NEW - 1:
            break
        with torch.autocast("cuda", dtype=torch.float16):
            if HAS_CACHE:
                logits, cache = model.decode_step(nxt[:, None], cache)
            else:
                logits = model(torch.cat([ids, gen], 1), plen)[:, -1]
    return [clean(tok.decode(g.tolist())) for g in gen]


# ----------------------------------------------------------------------------- run
data = json.load(open(IN_PATH, encoding="utf-8"))
all_ids = list(data.keys())
if RUN.get("smoke_test"):
    all_ids = all_ids[:16]
mine = all_ids[RANK::WORLD]
enc = {k: tok.encode(data[k]["text"]).ids[-PROMPT_MAX:] for k in mine}
order = sorted(mine, key=lambda k: len(enc[k]))
BS = int(GEN["batch_size"])
out, dropped = {}, 0
t0 = time.time()
for i in range(0, len(order), BS):
    keys = order[i:i + BS]
    m = min(len(enc[k]) for k in keys)
    dropped += sum(len(enc[k]) - m for k in keys)
    texts = generate_batch([enc[k][-m:] for k in keys])
    for k, t in zip(keys, texts):
        out[k] = t
    if RANK == 0 and (i // BS) % 5 == 0:
        el = time.time() - t0
        print(f"  rank0 {i+len(keys)}/{len(order)} prompts | {el/60:.1f} min | "
              f"eta {el/(i+len(keys))*(len(order)-i-len(keys))/60:.1f} min", flush=True)

part = f"{OUT_PREFIX}.part{RANK}.json"
json.dump(out, open(part, "w", encoding="utf-8"), ensure_ascii=False)
log(f"Context tokens trimmed for batching (rank0): {dropped} total")
if WORLD > 1:
    dist.barrier()

if RANK == 0:
    merged = {}
    for pf in sorted(glob.glob(f"{OUT_PREFIX}.part*.json")):
        merged.update(json.load(open(pf, encoding="utf-8")))
        os.remove(pf)
    preds, short = {}, 0
    for k in all_ids:
        w = merged.get(k, "").split()
        if len(w) < TARGET_WORDS:
            short += 1
        preds[k] = {"text": " ".join(w[:TARGET_WORDS])}
    pred_path = f"{OUT_PREFIX}_prediction.json"
    json.dump(preds, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    zip_path = f"{OUT_PREFIX}_submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")
    print(f"Wrote {pred_path} and {zip_path} | {len(preds)} ids | {short} under {TARGET_WORDS} words | "
          f"{(time.time()-t0)/60:.1f} min", flush=True)
if WORLD > 1:
    dist.destroy_process_group()


## Smoke test (≈5 min) — run before every real run
Uses `dummy_model.py` and tiny pipeline-test settings for anything still `TODO` in TRAIN/GEN
(these are **not** recommendations for your model). Checks: data loading, DDP, checkpoint save/resume,
validation, generation on 16 prompts, zip creation.

In [ ]:
# 12 — smoke test
import copy
smoke = copy.deepcopy(CFG)
PIPELINE_TEST_ONLY = dict(ctx_len=512, micro_batch=4, grad_accum=2, lr_max=1e-3, lr_min=1e-4, warmup_steps=10,
                          schedule="cosine", weight_decay=0.0, wd_exclude_1d=True, beta1=0.9, beta2=0.95,
                          grad_clip=1.0, prefix_mode="random", prefix_frac_min=0.25, prefix_frac_max=0.75,
                          val_prefix_mode="fixed", val_prefix_frac=0.5, mix="proportional")
for k, v in PIPELINE_TEST_ONLY.items():
    if smoke["train"].get(k) == TODO: smoke["train"][k] = v
for k, v in dict(checkpoint="final", max_new_tokens=200, batch_size=8, temperature=0, top_k=0, top_p=1.0,
                 rep_penalty=1.0, no_repeat_ngram=3).items():
    if smoke["gen"].get(k) == TODO: smoke["gen"][k] = v
smoke["run"].update(smoke_test=True, smoke_train_minutes=2, out_dir="/kaggle/working/smoke", run_name="smoke",
                    val_windows_per_source=8)
smoke["train"].update(val_every_min=1, ckpt_every_min=1)
json.dump(smoke, open("smoke_config.json", "w"))
shutil.rmtree("/kaggle/working/smoke", ignore_errors=True)
!timeout -k 30 600 torchrun --standalone --nproc_per_node={NGPU} train.py smoke_config.json
print("---- resume check (should say 'Resumed from') ----")
smoke["run"]["smoke_train_minutes"] = 0.5; json.dump(smoke, open("smoke_config.json", "w"))
!timeout -k 30 600 torchrun --standalone --nproc_per_node={NGPU} train.py smoke_config.json
!timeout -k 30 600 torchrun --standalone --nproc_per_node={NGPU} generate.py smoke_config.json {RUN['token_dir']}/local_test.json /kaggle/working/smoke/local
!ls -la /kaggle/working/smoke
_ok = os.path.exists("/kaggle/working/smoke/smoke_final.pt") and os.path.exists("/kaggle/working/smoke/local_submission.zip")
assert _ok, "SMOKE TEST FAILED — read the torchrun output above; the real run was not started"
print("SMOKE TEST PASSED")

## Real run
Trains until `session_hours − reserve_minutes_after_train`, then saves `<run>_final.pt` (and `<run>_best.pt` by validation loss).

In [ ]:
# 14 — train (your model.py, your config)
assert not find_todos(CFG), "fill every TODO first"
json.dump(CFG, open("run_config.json", "w"), indent=1)
_cap = int(SESSION_START + RUN["session_hours"] * 3600 - RUN["reserve_minutes_after_train"] * 60 - time.time()) + 1200
!timeout -k 60 {_cap} torchrun --standalone --nproc_per_node={NGPU} train.py run_config.json
_best = os.path.join(RUN["out_dir"], f"{RUN['run_name']}_best.pt")
_final = os.path.join(RUN["out_dir"], f"{RUN['run_name']}_final.pt")
assert os.path.exists(_best), "TRAINING FAILED — no checkpoint at all; see output above"
if not os.path.exists(_final):
    print("WARNING: training did not finish cleanly (no final.pt) — generating from best.pt")
    if GEN["checkpoint"] == "final": GEN["checkpoint"] = CFG["gen"]["checkpoint"] = "best"; json.dump(CFG, open("run_config.json", "w"), indent=1)
print("training finished:", _best)

def minutes_left():
    return (SESSION_START + RUN["session_hours"] * 3600 - time.time()) / 60

In [ ]:
# 15 — local evaluation on held-out pairs (prediction file -> official evaluation script)
OUT = RUN["out_dir"]
DO_LOCAL = minutes_left() > 25
print(f"{minutes_left():.0f} min left | local eval: {DO_LOCAL}")
if DO_LOCAL:
    !timeout -k 30 1500 torchrun --standalone --nproc_per_node={NGPU} generate.py run_config.json {RUN['token_dir']}/local_test.json {OUT}/local
    if not os.path.exists(f"{OUT}/local_prediction.json"):
        print("local generation failed — skipping local eval/sweep, going straight to the test set")
        DO_LOCAL = False
if DO_LOCAL:


    def rouge_l_f1(a, b):
        a, b = a.split(), b.split()
        if not a or not b: return 0.0
        prev = [0] * (len(b) + 1)
        for x in a:
            cur = [0]
            for j, y in enumerate(b):
                cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[-1]))
            prev = cur
        l = prev[-1]
        if l == 0: return 0.0
        p, r = l / len(a), l / len(b)
        return 2 * p * r / (p + r)

    pred = json.load(open(f"{OUT}/local_prediction.json")); ref = json.load(open(f"{RUN['token_dir']}/local_ref.json"))
    origin = json.load(open(f"{RUN['token_dir']}/local_origin.json"))
    scores = {k: rouge_l_f1(pred[k]["text"], ref[k]["text"]) for k in ref}
    print(f"ROUGE-L F1 (local, n={len(scores)}): {sum(scores.values())/len(scores):.4f}")
    by_src = {}
    for k, s in scores.items(): by_src.setdefault(origin[k], []).append(s)
    for s, v in by_src.items(): print(f"  {s:15s} {sum(v)/len(v):.4f}  (n={len(v)})")
    print(f"Official scorer: prediction={OUT}/local_prediction.json  reference={RUN['token_dir']}/local_ref.json")

In [ ]:
# 15b — decoding sweep on the local held-out pairs; keeps the best setting for the test run (cell 16)
if not DO_LOCAL:
    print('skipping decoding sweep (no time); test generation uses GEN from the config cell')
else:
    import copy, statistics
    CANDS = {
        "greedy_nr3_rp1.1": dict(temperature=0, top_k=0, top_p=1.0, rep_penalty=1.1, no_repeat_ngram=3),
        "greedy_rp1.2":     dict(temperature=0, top_k=0, top_p=1.0, rep_penalty=1.2, no_repeat_ngram=0),
        "sample_t0.7":      dict(temperature=0.7, top_k=50, top_p=0.9, rep_penalty=1.1, no_repeat_ngram=4),
    }
    results = {"config (cell 15)": (statistics.mean(scores.values()), {})}
    for name, over in CANDS.items():
        left = (SESSION_START + RUN["session_hours"] * 3600 - time.time()) / 60
        if left < 30:
            print(f"skipping {name}: {left:.0f} min left, keeping time for the test set"); break
        c = copy.deepcopy(CFG); c["gen"].update(over); json.dump(c, open(f"gen_{name}.json", "w"))
        !timeout -k 30 1200 torchrun --standalone --nproc_per_node={NGPU} generate.py gen_{name}.json {RUN['token_dir']}/local_test.json {OUT}/sweep_{name}
        if not os.path.exists(f"{OUT}/sweep_{name}_prediction.json"):
            print(f"{name} failed, skipped"); continue
        p = json.load(open(f"{OUT}/sweep_{name}_prediction.json"))
        results[name] = (statistics.mean(rouge_l_f1(p[k]["text"], ref[k]["text"]) for k in ref), over)
    for k, (s, _) in sorted(results.items(), key=lambda kv: -kv[1][0]):
        print(f"  {k:20s} ROUGE-L {s:.4f}")
    best = max(results, key=lambda k: results[k][0])
    CFG["gen"].update(results[best][1]); GEN = CFG["gen"]
    json.dump(CFG, open("run_config.json", "w"), indent=1)
    print("test generation will use:", best, {k: GEN[k] for k in ("temperature", "top_k", "top_p", "rep_penalty", "no_repeat_ngram")})

In [ ]:
# 16 — official test set -> submission zip
_cap = max(600, int(SESSION_START + RUN['session_hours'] * 3600 - time.time()) - 120)
!timeout -k 30 {_cap} torchrun --standalone --nproc_per_node={NGPU} generate.py run_config.json {TEST_PATH} {RUN['out_dir']}/test
_zip = os.path.join(RUN["out_dir"], "test_submission.zip")
assert os.path.exists(_zip), "TEST GENERATION FAILED — see output above"
shutil.copy(_zip, "/kaggle/working/submission.zip")
shutil.copy(os.path.join(RUN["out_dir"], "test_prediction.json"), "/kaggle/working/prediction.json")
print("ready: /kaggle/working/submission.zip")

In [ ]:
# 17 — validate the zip before uploading
import zipfile
def validate_zip(zip_path, test_path):
    test = json.load(open(test_path, encoding="utf-8"))
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"zip must contain only prediction.json at the root, has {names}"
        pred = json.loads(z.read("prediction.json").decode("utf-8"))
    missing, extra = set(test) - set(pred), set(pred) - set(test)
    assert not missing, f"{len(missing)} test IDs missing, e.g. {list(missing)[:5]}"
    assert not extra, f"{len(extra)} unknown IDs, e.g. {list(extra)[:5]}"
    bad = [k for k, v in pred.items() if not isinstance(v, dict) or not isinstance(v.get("text"), str)]
    assert not bad, f"bad entries: {bad[:5]}"
    wc = [len(v["text"].split()) for v in pred.values()]
    empty = sum(w == 0 for w in wc)
    leaked = sum(pred[k]["text"][:200] in test[k]["text"] for k in pred if pred[k]["text"])
    print(f"OK: {len(pred)} ids | words min/mean/max {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)} | "
          f"empty {empty} | outputs that copy prompt text {leaked}")
validate_zip("/kaggle/working/submission.zip", TEST_PATH)
from IPython.display import FileLink; display(FileLink("submission.zip"))

In [ ]:
# 18 — training curves for the report
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f"{RUN['out_dir']}/{RUN['run_name']}_log.csv")
tr, va = log[log.train_loss.notna()], log[log.val_loss.notna()]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tr.tokens / 1e6, tr.train_loss, lw=0.8, label="train"); ax[0].plot(va.tokens / 1e6, va.val_loss, "o-", label="val")
ax[0].set_xlabel("tokens (M)"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(RUN["run_name"])
ax[1].plot(tr.tokens / 1e6, tr.tok_per_s / 1e3); ax[1].set_xlabel("tokens (M)"); ax[1].set_ylabel("K tokens/s")
plt.tight_layout(); plt.savefig(f"{RUN['out_dir']}/{RUN['run_name']}_curves.png", dpi=120); plt.show()

In [ ]:
# 23 — trim /kaggle/working so this version's output stays small and mounts reliably
import glob
for d in ["/kaggle/working/smoke"]:
    shutil.rmtree(d, ignore_errors=True)
for f in glob.glob(f"{RUN['out_dir']}/sweep_*") + glob.glob("/kaggle/working/gen_*.json"):
    os.remove(f)
_steps = sorted(glob.glob(f"{RUN['out_dir']}/{RUN['run_name']}_step*.pt"), key=lambda q: int(q.split("_step")[-1][:-3]))
for f in _steps[:-1]:
    os.remove(f)
tot = 0
for root, _, files in os.walk("/kaggle/working"):
    for f in files:
        q = os.path.join(root, f); s = os.path.getsize(q); tot += s
        if s > 50e6: print(f"{s/1e9:6.2f} GB  {q}")
print(f"output total: {tot/1e9:.2f} GB")